# Multi-Currency Curve Trace Demo

This notebook compares the generic curve tracer across multiple currencies and a USD dual-curve setup.

It uses the same ORE fixture and shows:
- discounting curve selection by currency
- dependency graph size by currency
- native interpolation nodes across currencies
- the USD 6M dual-curve dependency chain (`USD6M -> USD3M -> USD1D`)


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and ROOT.name != "Tools":
    ROOT = ROOT.parent
if ROOT.name != "Tools":
    raise RuntimeError("Run this notebook from somewhere under the Engine workspace")

PYRUNNER = ROOT / "PythonOreRunner"
if str(PYRUNNER) not in sys.path:
    sys.path.insert(0, str(PYRUNNER))

from ore_curve_fit_parity import trace_discount_curve_from_ore, trace_index_curve_from_ore

ORE_XML = PYRUNNER / "parity_artifacts" / "multiccy_benchmark_final" / "cases" / "flat_USD_5Y_B" / "Input" / "ore.xml"
CCYS = ["EUR", "GBP", "CHF", "JPY", "USD"]


In [ ]:
discount_traces = {ccy: trace_discount_curve_from_ore(ORE_XML, currency=ccy) for ccy in CCYS}
usd6m_trace = trace_index_curve_from_ore(ORE_XML, index_name="USD-LIBOR-6M")

summary_rows = []
for ccy, trace in discount_traces.items():
    summary_rows.append(
        {
            "currency": ccy,
            "curve_handle": trace["curve_handle"],
            "curve_name": trace["curve_name"],
            "curve_id": trace["curve_config"]["curve_id"],
            "dependency_graph_size": len(trace["dependency_graph"]),
            "native_node_count": len(trace["ore_calibration_trace"]["pillars"]),
            "report_grid_count": len(trace["ore_curve_points"]["times"]),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(summary_df["currency"], summary_df["native_node_count"], color="#457b9d")
axes[0].set_title("Native interpolation nodes by currency")
axes[0].set_ylabel("Node count")

axes[1].bar(summary_df["currency"], summary_df["dependency_graph_size"], color="#2a9d8f")
axes[1].set_title("Dependency graph size by currency")
axes[1].set_ylabel("Curve count in graph")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
palette = {
    "EUR": "#264653",
    "GBP": "#2a9d8f",
    "CHF": "#e9c46a",
    "JPY": "#f4a261",
    "USD": "#e76f51",
}

for ccy, trace in discount_traces.items():
    nodes = pd.DataFrame(trace["ore_calibration_trace"]["pillars"])
    nodes["time"] = pd.to_numeric(nodes["time"], errors="coerce")
    nodes["discountFactor"] = pd.to_numeric(nodes["discountFactor"], errors="coerce")
    ax.scatter(nodes["time"], nodes["discountFactor"], s=28, label=ccy, color=palette[ccy])

ax.set_title("Native interpolation nodes across currencies")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Discount factor")
ax.grid(True, alpha=0.3)
ax.legend(ncol=3)
plt.show()


In [ ]:
usd6m_graph_rows = []
for curve_id, payload in usd6m_trace["dependency_graph"].items():
    usd6m_graph_rows.append(
        {
            "curve_id": curve_id,
            "dependencies": ", ".join(payload["dependencies"]),
            "segment_count": len(payload["curve_config"]["segments"]),
        }
    )

pd.DataFrame(usd6m_graph_rows).sort_values("curve_id")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis("off")

boxes = [
    (0.15, "USD1D", "discount anchor"),
    (0.50, "USD3M", "depends on USD1D"),
    (0.85, "USD6M", "depends on USD1D + USD3M"),
]
for x, title, detail in boxes:
    ax.text(
        x,
        0.5,
        f"{title}\n{detail}",
        ha="center",
        va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5", fc="#eef3f7", ec="#355070", lw=1.5),
        transform=ax.transAxes,
    )

for start, end in [(0.24, 0.41), (0.59, 0.76)]:
    ax.annotate(
        "",
        xy=(end, 0.5),
        xytext=(start, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2, color="#355070"),
        xycoords=ax.transAxes,
    )

ax.text(0.67, 0.68, "direct dependency", fontsize=9, color="#355070", transform=ax.transAxes)
ax.text(0.33, 0.68, "direct dependency", fontsize=9, color="#355070", transform=ax.transAxes)
ax.set_title("USD 6M dual-curve dependency chain")
plt.show()
